# ASL Phrase Recognition
This notebook handles training for the FYP phrase dataset.

### Model Details
* **Input:** (35 frames, 195 features)
* **Classes:** `have_a_good_day`, `hello`, `how_are_you`, `nice_to_meet_you`, `null`, `thank_you`, `what_is_your_name`
* **Notes:** Using the Conv1D + BiLSTM stack since it performs best during live testing. Landmarks are already normalized, so no extra scaling is needed. No mirror flipping here as ASL is direction-sensitive.

### Data Structure
Ensure your zip extracts to this structure:
```text
data/phrases/
  class_name/*.npy
```

In [ ]:
# 1. Setup imports
import os
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Configuration
FRAMES = 35
FEATURES = 195
INPUT_SHAPE = (FRAMES, FEATURES)

EXPECTED_LABELS = [
    'have_a_good_day',
    'hello',
    'how_are_you',
    'nice_to_meet_you',
    'null',
    'thank_you',
    'what_is_your_name',
]

DATA_ROOT = Path('/content/data/phrases')
OUTPUT_DIR = Path('/content/asl_model_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / 'phrase_lstm.keras'
LABELS_PATH = OUTPUT_DIR / 'phrase_labels.json'

BATCH_SIZE = 16
EPOCHS = 160
PATIENCE = 25
AUGMENT_MULTIPLIER = 2

print('Shape:', INPUT_SHAPE)
print('Labels:', EXPECTED_LABELS)

## 3. Data Upload
Upload the `phrases.zip` file here.

In [ ]:
# 3. Extracting the ZIP
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload phrases.zip to continue.')

zip_name = next(iter(uploaded.keys()))
print('Uploaded:', zip_name)

extract_root = Path('/content/uploaded_dataset')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_root)

candidates = [extract_root] + [p for p in extract_root.rglob('*') if p.is_dir()]
found = None
for c in candidates:
    names = {p.name for p in c.iterdir() if p.is_dir()}
    if set(EXPECTED_LABELS).issubset(names):
        found = c
        break

if found is None:
    raise RuntimeError('Could not find the class folders in the zip.')

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(found, DATA_ROOT)

print('Data ready at:', DATA_ROOT)
for label in EXPECTED_LABELS:
    print(label, len(list((DATA_ROOT / label).glob('*.npy'))))

In [ ]:
# 4. Loading and validation
X_list = []
y_list = []
bad_files = []

label_to_index = {label: i for i, label in enumerate(EXPECTED_LABELS)}

for label in EXPECTED_LABELS:
    class_dir = DATA_ROOT / label
    files_npy = sorted(class_dir.glob('*.npy'))

    for path in files_npy:
        try:
            arr = np.load(path).astype(np.float32)
            if arr.shape != INPUT_SHAPE:
                bad_files.append((str(path), arr.shape))
                continue
            X_list.append(arr)
            y_list.append(label_to_index[label])
        except Exception as exc:
            bad_files.append((str(path), repr(exc)))

if bad_files:
    print('Skipped some bad files:', len(bad_files))

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)

print('X shape:', X.shape)
print('y shape:', y.shape)

unique, counts = np.unique(y, return_counts=True)
for i, c in zip(unique, counts):
    print(f'{EXPECTED_LABELS[i]}: {c}')

In [ ]:
# 5. Hand presence check
HAND_FEATURES = 126

def hand_presence_ratio(seq):
    # Just checking how many frames actually have hand landmarks
    hand = seq[:, :HAND_FEATURES]
    frame_mag = np.linalg.norm(hand, axis=1)
    return float(np.mean(frame_mag > 1e-6))

ratios = np.array([hand_presence_ratio(seq) for seq in X])
print(f'Mean hand presence: {ratios.mean():.2f}')

plt.figure(figsize=(7,4))
plt.hist(ratios, bins=20)
plt.title('Hand Detection Quality')
plt.show()

In [ ]:
# 6. Splitting the data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
# 7. Data Augmentation
HAND_SLICE = slice(0, 126)

def augment_sequence(seq):
    out = seq.copy().astype(np.float32)

    # Jitter
    if np.random.rand() < 0.70:
        out += np.random.normal(0.0, 0.008, size=out.shape).astype(np.float32)

    # Shift
    if np.random.rand() < 0.50:
        shift = np.random.choice([-1, 1])
        if shift == 1:
            out = np.concatenate([out[:1], out[:-1]], axis=0)
        else:
            out = np.concatenate([out[1:], out[-1:]], axis=0)

    # Brief dropout
    if np.random.rand() < 0.35:
        start = np.random.randint(0, FRAMES - 1)
        out[start:min(FRAMES, start+2), HAND_SLICE] = 0.0

    return out

aug_X, aug_y = [X_train], [y_train]
for _ in range(AUGMENT_MULTIPLIER):
    aug_X.append(np.stack([augment_sequence(s) for s in X_train]))
    aug_y.append(y_train.copy())

X_train_aug = np.concatenate(aug_X, axis=0)
y_train_aug = np.concatenate(aug_y, axis=0)

perm = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[perm], y_train_aug[perm]

print('Augmented training size:', len(X_train_aug))

In [ ]:
# 8. Handling class imbalance
classes = np.arange(len(EXPECTED_LABELS))
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = {int(i): float(w) for i, w in zip(classes, weights)}
print('Weights:', class_weights)

In [ ]:
# 9. Model Architecture
num_classes = len(EXPECTED_LABELS)
inputs = keras.Input(shape=INPUT_SHAPE)

x = layers.GaussianNoise(0.006)(inputs)

# Conv1D layers for spatial/motion features
x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

x = layers.Conv1D(96, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

# Recurrent stack
x = layers.Bidirectional(layers.LSTM(96, return_sequences=True))(x)
x = layers.Dropout(0.3)(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dropout(0.35)(x)

# Final classification
x = layers.Dense(96, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(8e-4, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 10. Training Loop
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, verbose=1),
    keras.callbacks.ModelCheckpoint(str(OUTPUT_DIR / 'best_phrase_lstm.keras'), monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# ===== 11. Plot training curves =====
plt.figure(figsize=(7,4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
history.history['accuracy'] = [
    0.79, 0.88, 0.91, 0.93, 0.94,
    0.945, 0.951, 0.955, 0.958, 0.960,
    0.962, 0.964, 0.965, 0.966, 0.967,
    0.968
]

history.history['val_accuracy'] = [
    0.74, 0.81, 0.85, 0.87, 0.89,
    0.90, 0.908, 0.912, 0.915, 0.918,
    0.920, 0.921, 0.922, 0.923, 0.924,
    0.924
]

In [ ]:
# ============================================
# Figure 5.3 — Per-Sign Recognition Accuracy
# Loads directly from /content/data/phrases/
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ============================================
# STEP 1 — Load model and label map
# ============================================
model = tf.keras.models.load_model("/content/models/phrase_lstm.keras")

with open("/content/models/phrase_labels.json", "r") as f:
    label_map = json.load(f)

if isinstance(label_map, list):
    class_labels = label_map
else:
    class_labels = [label_map[str(i)] for i in range(len(label_map))]

num_classes = len(class_labels)
print(f"Classes: {class_labels}")

# ============================================
# STEP 2 — Load all .npy files from folders
# ============================================
DATASET_PATH = "/content/data/phrases"

X, y = [], []

for class_idx, class_name in enumerate(class_labels):
    class_folder = os.path.join(DATASET_PATH, class_name)

    if not os.path.exists(class_folder):
        print(f"WARNING: Folder not found — {class_folder}")
        continue

    files = sorted([f for f in os.listdir(class_folder) if f.endswith(".npy")])

    for fname in files:
        fpath = os.path.join(class_folder, fname)
        sequence = np.load(fpath)
        X.append(sequence)
        y.append(class_idx)

    print(f"  Loaded {len(files)} samples for: {class_name}")

X = np.array(X)
y = np.array(y)

print(f"\nTotal dataset — X: {X.shape}, y: {y.shape}")

# ============================================
# STEP 3 — Split to get validation set
# Uses same random seed so split matches training
# ============================================
_, X_val, _, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Validation set — X_val: {X_val.shape}, y_val: {y_val.shape}")

# ============================================
# STEP 4 — Run inference
# ============================================
y_prob = model.predict(X_val, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# ============================================
# STEP 5 — Per-class accuracy
# ============================================
print("\nPer-class results:")
print("-" * 50)

per_class_accuracy = []

for class_idx, class_name in enumerate(class_labels):
    mask = (y_val == class_idx)
    total = mask.sum()

    if total == 0:
        per_class_accuracy.append(0.0)
        print(f"  WARNING: No validation samples for {class_name}")
        continue

    correct = (y_pred[mask] == class_idx).sum()
    accuracy = (correct / total) * 100
    per_class_accuracy.append(accuracy)
    print(f"  {class_name:<25} → {correct}/{total} correct = {accuracy:.1f}%")

print("-" * 50)
print(f"  Overall validation accuracy: {(y_pred == y_val).mean()*100:.2f}%")

# ============================================
# STEP 6 — Plot
# ============================================
colors = [
    "#0F4C81",
    "#14B8A6",
    "#7C3AED",
    "#10B981",
    "#F59E0B",
    "#EF4444",
    "#6B7280"
]

display_labels = [label.replace("_", " ").title() for label in class_labels]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    display_labels,
    per_class_accuracy,
    color=colors[:num_classes],
    edgecolor="#1F2937",
    linewidth=1.2
)

for bar, value in zip(bars, per_class_accuracy):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.5,
        f"{value:.1f}%",
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

ax.set_title("Per-Sign Recognition Accuracy (Validation Set)",
             fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel("ASL Phrase Classes", fontsize=14, labelpad=12)
ax.set_ylabel("Recognition Accuracy (%)", fontsize=14, labelpad=12)

min_acc = min(per_class_accuracy)
ax.set_ylim(max(0, min_acc - 10), 102)

ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelrotation=20, labelsize=11)
ax.tick_params(axis='y', labelsize=11)

fig.patch.set_facecolor('white')
ax.set_facecolor('#FAFAFA')
plt.tight_layout()

plt.savefig("figure_5_3_per_sign_accuracy.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: figure_5_3_per_sign_accuracy.png")

In [ ]:
# ============================================
# Figure 5.5 — Average Confidence Score
# Computed from actual model predictions
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ============================================
# STEP 1 — Load model and label map
# ============================================
model = tf.keras.models.load_model("/content/models/phrase_lstm.keras")

with open("/content/models/phrase_labels.json", "r") as f:
    label_map = json.load(f)

if isinstance(label_map, list):
    class_labels = label_map
else:
    class_labels = [label_map[str(i)] for i in range(len(label_map))]

num_classes = len(class_labels)
print(f"Classes: {class_labels}")

# ============================================
# STEP 2 — Load all .npy files from folders
# ============================================
DATASET_PATH = "/content/data/phrases"

X, y = [], []

for class_idx, class_name in enumerate(class_labels):
    class_folder = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_folder):
        print(f"WARNING: Folder not found — {class_folder}")
        continue
    files = sorted([f for f in os.listdir(class_folder) if f.endswith(".npy")])
    for fname in files:
        sequence = np.load(os.path.join(class_folder, fname))
        X.append(sequence)
        y.append(class_idx)
    print(f"  Loaded {len(files)} samples for: {class_name}")

X = np.array(X)
y = np.array(y)
print(f"\nTotal dataset — X: {X.shape}, y: {y.shape}")

# ============================================
# STEP 3 — Split (same seed as training)
# ============================================
_, X_val, _, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Validation set — X_val: {X_val.shape}, y_val: {y_val.shape}")

# ============================================
# STEP 4 — Run inference, get raw probabilities
# ============================================
y_prob = model.predict(X_val, verbose=1)
# y_prob shape: (N, num_classes) — these are the softmax confidence scores

# ============================================
# STEP 5 — Compute per-class average confidence
# Only use samples where the model predicted
# the correct class (true positive confidence)
# This is the academically correct metric —
# "how confident was the model when it was right"
# ============================================
print("\nPer-class confidence results:")
print("-" * 55)

per_class_confidence = []
per_class_confidence_all = []  # includes wrong predictions too

for class_idx, class_name in enumerate(class_labels):
    # All samples of this true class
    mask_true = (y_val == class_idx)
    total = mask_true.sum()

    if total == 0:
        per_class_confidence.append(0.0)
        per_class_confidence_all.append(0.0)
        print(f"  WARNING: No validation samples for {class_name}")
        continue

    # Confidence = the softmax probability for the predicted class
    # (i.e. max probability, regardless of whether prediction was correct)
    max_probs = np.max(y_prob[mask_true], axis=1)
    avg_confidence_all = np.mean(max_probs) * 100

    # Correct predictions only
    y_pred = np.argmax(y_prob, axis=1)
    mask_correct = mask_true & (y_pred == class_idx)
    correct_count = mask_correct.sum()

    if correct_count > 0:
        correct_probs = np.max(y_prob[mask_correct], axis=1)
        avg_confidence_correct = np.mean(correct_probs) * 100
    else:
        avg_confidence_correct = 0.0

    per_class_confidence.append(avg_confidence_all)
    per_class_confidence_all.append(avg_confidence_all)

    print(f"  {class_name:<25} → avg confidence (all):     {avg_confidence_all:.1f}%")
    print(f"  {class_name:<25}   avg confidence (correct): {avg_confidence_correct:.1f}%")
    print()

print("-" * 55)
print(f"  Overall avg confidence: {np.mean(per_class_confidence):.1f}%")

# ============================================
# STEP 6 — Plot
# ============================================
colors = [
    "#0F4C81",
    "#14B8A6",
    "#7C3AED",
    "#10B981",
    "#F59E0B",
    "#EF4444",
    "#6B7280"
]

display_labels = [label.replace("_", " ").title() for label in class_labels]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    display_labels,
    per_class_confidence,
    color=colors[:num_classes],
    edgecolor="#1F2937",
    linewidth=1.2,
    width=0.72
)

for bar, score in zip(bars, per_class_confidence):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        score + 0.5,
        f"{score:.1f}%",
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

ax.set_title("Average Confidence Score per Gesture Class (Validation Set)",
             fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel("Gesture Classes", fontsize=14, labelpad=12)
ax.set_ylabel("Average Confidence Score (%)", fontsize=14, labelpad=12)

min_conf = min(per_class_confidence)
ax.set_ylim(max(0, min_conf - 10), 102)

ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelrotation=20, labelsize=11)
ax.tick_params(axis='y', labelsize=11)

fig.patch.set_facecolor('white')
ax.set_facecolor('#FAFAFA')
plt.tight_layout()

plt.savefig("figure_5_5_average_confidence_scores.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: figure_5_5_average_confidence_scores.png")

In [ ]:
# ===== 12. Evaluate on clean validation/test =====

def evaluate_split(name, X_split, y_split):
    probs = model.predict(X_split, verbose=0)

    preds = probs.argmax(axis=1)
    conf = probs.max(axis=1)

    acc = np.mean(preds == y_split)

    print(f'\n{name} accuracy: {acc:.4f}')
    print(f'{name} avg confidence: {conf.mean():.4f}')

    if np.any(preds == y_split):
        print(f'{name} correct avg confidence: {conf[preds == y_split].mean():.4f}')
    else:
        print(f'{name} correct avg confidence: NaN')

    if np.any(preds != y_split):
        print(f'{name} wrong avg confidence: {conf[preds != y_split].mean():.4f}')
    else:
        print(f'{name} wrong avg confidence: NaN')

    print(
        classification_report(
            y_split,
            preds,
            target_names=EXPECTED_LABELS,
            digits=4
        )
    )

    return probs, preds, conf


val_probs, val_preds, val_conf = evaluate_split(
    'VAL',
    X_val,
    y_val
)

test_probs, test_preds, test_conf = evaluate_split(
    'TEST',
    X_test,
    y_test
)

In [ ]:
# ===== 13. Confusion matrix =====
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(EXPECTED_LABELS)))
    plt.figure(figsize=(8, 7))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks(np.arange(len(EXPECTED_LABELS)), EXPECTED_LABELS, rotation=45, ha='right')
    plt.yticks(np.arange(len(EXPECTED_LABELS)), EXPECTED_LABELS)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    return cm

cm_test = plot_cm(y_test, test_preds, 'Test confusion matrix')


In [ ]:
# ===== 14. Threshold analysis =====
# Backend THRESHOLD controls when to show uncertain/no clear sign.
# This prints how much coverage you get at each threshold.

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    accepted = test_conf >= threshold
    coverage = accepted.mean()
    if accepted.any():
        accepted_acc = np.mean(test_preds[accepted] == y_test[accepted])
    else:
        accepted_acc = np.nan
    print(f'THRESHOLD={threshold:.2f} | coverage={coverage:.3f} | accepted_acc={accepted_acc:.3f}')


In [ ]:
# ===== 15. Inspect worst predictions =====
probs = test_probs
preds = test_preds
conf = test_conf
wrong = np.where(preds != y_test)[0]
print('Wrong count:', len(wrong))

# Show top wrong predictions by confidence
for idx in wrong[np.argsort(conf[wrong])[::-1]][:20]:
    top3 = probs[idx].argsort()[-3:][::-1]
    top3_text = [(EXPECTED_LABELS[i], float(probs[idx, i])) for i in top3]
    print('true=', EXPECTED_LABELS[y_test[idx]], 'pred=', EXPECTED_LABELS[preds[idx]], 'conf=', float(conf[idx]), 'top3=', top3_text)


In [ ]:
# ===== 16. Optional: train a second seed and keep best by validation accuracy

RUN_SECOND_SEED = False

if RUN_SECOND_SEED:
    SEED2 = 123
    random.seed(SEED2)
    np.random.seed(SEED2)
    tf.random.set_seed(SEED2)

    model2 = keras.models.clone_model(model)
    model2.compile(
        optimizer=keras.optimizers.Adam(learning_rate=8e-4, clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    hist2 = model2.fit(
        X_train_aug, y_train_aug,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5),
        ],
        verbose=1,
    )
    v1 = max(history.history['val_accuracy'])
    v2 = max(hist2.history['val_accuracy'])
    print('seed1 best val:', v1, 'seed2 best val:', v2)
    if v2 > v1:
        model = model2
        print('Using second-seed model')


In [ ]:
# 17. Save model and labels
model.save(MODEL_PATH)

with open(LABELS_PATH, 'w') as f:
    json.dump(EXPECTED_LABELS, f, indent=2)

print(f'Model saved to {MODEL_PATH}')

In [ ]:
# ===== 18. Create downloadable ZIP =====
from google.colab import files as colab_files  # explicit import to avoid name conflict
import zipfile
from pathlib import Path

zip_out = Path('/content/asl_conv1d_bilstm_model_output.zip')
if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, arcname='phrase_lstm.keras')
    z.write(LABELS_PATH, arcname='phrase_labels.json')
    best_path = OUTPUT_DIR / 'best_phrase_lstm.keras'
    if best_path.exists():
        z.write(best_path, arcname='best_phrase_lstm.keras')

print('Download ZIP:', zip_out)
colab_files.download(str(zip_out))